# Zero-Shot Transfer Evaluation on INbreast

**Evaluate optimized models on the target dataset without fine-tuning**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dtobi59/mammography-multiobjective-optimization/blob/main/zero_shot_evaluation.ipynb)

This notebook:
1. Loads Pareto-optimal models from NSGA-III optimization
2. Evaluates them on INbreast (zero-shot, no fine-tuning)
3. Reports image-level and breast-level metrics
4. Compares transfer performance across solutions
5. Saves results to Google Drive

**Prerequisites:**
- Completed NSGA-III optimization (colab_tutorial.ipynb)
- Model checkpoints saved in Google Drive
- Pareto solutions CSV file available

**Author:** David ([@dtobi59](https://github.com/dtobi59))

## 1. Setup Environment

Check GPU and clone repository.

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone the repository
!git clone https://github.com/dtobi59/mammography-multiobjective-optimization.git

# Change to project directory
%cd mammography-multiobjective-optimization

# List files
!ls -la

In [ ]:
# Install required packages
!pip install -q -r requirements.txt

print("\n[SUCCESS] All dependencies installed!")

In [ ]:
# Setup Python path
import sys
import os

project_root = os.getcwd()
print(f"Project root: {project_root}")

if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Added {project_root} to sys.path")

print(f"\nPython sys.path[0]: {sys.path[0]}")
print("[OK] Path setup complete!")

## 2. Mount Google Drive

Access your optimization results and INbreast dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set paths to your data in Google Drive
INBREAST_PATH = "/content/drive/MyDrive/INbreast"
OPTIMIZATION_DIR = "/content/drive/MyDrive/vindr_optimization"
CHECKPOINT_DIR = f"{OPTIMIZATION_DIR}/checkpoints"
RESULTS_DIR = f"{OPTIMIZATION_DIR}/results"

print("\n[SUCCESS] Google Drive mounted!")
print(f"INbreast dataset: {INBREAST_PATH}")
print(f"Optimization results: {OPTIMIZATION_DIR}")

## 3. Load Optimization Results

Load the Pareto front from your NSGA-III optimization.

In [ ]:
import glob
import pandas as pd
from pathlib import Path

# Find Pareto solutions CSV files
pareto_files = sorted(glob.glob(f"{RESULTS_DIR}/pareto_solutions_*.csv"))

print("=" * 80)
print("AVAILABLE OPTIMIZATION RESULTS")
print("=" * 80)

if not pareto_files:
    print("[ERROR] No Pareto solutions found!")
    print(f"\nExpected location: {RESULTS_DIR}")
    print("\nPlease run the optimization notebook first (colab_tutorial.ipynb)")
else:
    print(f"Found {len(pareto_files)} result file(s):\n")
    for i, file in enumerate(pareto_files):
        filename = Path(file).name
        print(f"  {i+1}. {filename}")
    print()
    
    # Load the most recent results
    latest_results = pareto_files[-1]
    print(f"Loading most recent: {Path(latest_results).name}")
    
    pareto_df = pd.read_csv(latest_results)
    print(f"\n[OK] Loaded {len(pareto_df)} Pareto-optimal solutions")
    print("=" * 80)

In [ ]:
# Display Pareto front summary
print("=" * 80)
print("PARETO FRONT SUMMARY")
print("=" * 80)
print(f"Total solutions: {len(pareto_df)}\n")

print("Best solutions for each objective:\n")

best_pr_auc_idx = pareto_df['pr_auc'].idxmax()
print(f"  Best PR-AUC:")
print(f"    Solution ID: {best_pr_auc_idx}")
print(f"    PR-AUC:  {pareto_df.loc[best_pr_auc_idx, 'pr_auc']:.4f}")
print(f"    AUROC:   {pareto_df.loc[best_pr_auc_idx, 'auroc']:.4f}")
print(f"    Brier:   {pareto_df.loc[best_pr_auc_idx, 'brier']:.4f}")
print()

best_auroc_idx = pareto_df['auroc'].idxmax()
print(f"  Best AUROC:")
print(f"    Solution ID: {best_auroc_idx}")
print(f"    PR-AUC:  {pareto_df.loc[best_auroc_idx, 'pr_auc']:.4f}")
print(f"    AUROC:   {pareto_df.loc[best_auroc_idx, 'auroc']:.4f}")
print(f"    Brier:   {pareto_df.loc[best_auroc_idx, 'brier']:.4f}")
print()

best_brier_idx = pareto_df['brier'].idxmin()
print(f"  Best Brier:")
print(f"    Solution ID: {best_brier_idx}")
print(f"    PR-AUC:  {pareto_df.loc[best_brier_idx, 'pr_auc']:.4f}")
print(f"    AUROC:   {pareto_df.loc[best_brier_idx, 'auroc']:.4f}")
print(f"    Brier:   {pareto_df.loc[best_brier_idx, 'brier']:.4f}")
print()

best_robust_idx = pareto_df['robustness_degradation'].idxmin()
print(f"  Best Robustness:")
print(f"    Solution ID: {best_robust_idx}")
print(f"    PR-AUC:  {pareto_df.loc[best_robust_idx, 'pr_auc']:.4f}")
print(f"    AUROC:   {pareto_df.loc[best_robust_idx, 'auroc']:.4f}")
print(f"    Brier:   {pareto_df.loc[best_robust_idx, 'brier']:.4f}")
print(f"    Robustness: {pareto_df.loc[best_robust_idx, 'robustness_degradation']:.4f}")

print("=" * 80)

# Display first few solutions
print("\nFirst 10 solutions:")
pareto_df.head(10)

## 4. Load INbreast Dataset

Load the target dataset for zero-shot evaluation.

In [ ]:
# Update config with INbreast path and correct image directory
import os
with open('config.py', 'r') as f:
    config_content = f.read()

# Update INbreast path
config_content = config_content.replace(
    'INBREAST_PATH = "/content/drive/MyDrive/INbreast"',
    f'INBREAST_PATH = "{INBREAST_PATH}"'
)

# IMPORTANT: Update image_dir to point to AllDICOMs (where DICOM files are)
# Later we'll convert to PNG and update this to images_png
if '"image_dir": "images"' in config_content:
    config_content = config_content.replace(
        '"image_dir": "images"',
        '"image_dir": "AllDICOMs"'
    )
    print("[OK] Updated image_dir to AllDICOMs")

with open('config.py', 'w') as f:
    f.write(config_content)

# Reload config module
import importlib
if 'config' in sys.modules:
    importlib.reload(config)
else:
    import config

print("[OK] Configuration updated!")
print(f"INbreast path: {INBREAST_PATH}")
print(f"Image directory: {config.INBREAST_CONFIG['image_dir']}")

# Verify the image directory exists and contains files
import os
from pathlib import Path
img_dir = Path(INBREAST_PATH) / config.INBREAST_CONFIG["image_dir"]
if img_dir.exists():
    files = list(img_dir.glob("*.*"))[:5]
    print(f"\n✓ Image directory exists: {img_dir}")
    print(f"  Total files: {len(list(img_dir.glob('*.*')))}")
    print(f"  Sample files:")
    for f in files:
        print(f"    - {f.name}")
else:
    print(f"\n⚠ Image directory not found: {img_dir}")

In [ ]:
import config
from optimization.nsga3_runner import load_metadata

print("=" * 80)
print("LOADING INBREAST DATASET")
print("=" * 80)

# Load INbreast metadata
inbreast_metadata = load_metadata(
    dataset_name="inbreast",
    dataset_path=config.INBREAST_PATH,
    dataset_config=config.INBREAST_CONFIG
)

print(f"\n[OK] Loaded {len(inbreast_metadata)} images")
print(f"Patients: {inbreast_metadata['patient_id'].nunique()}")
print(f"Breasts: {inbreast_metadata['breast_id'].nunique()}")
print()
print("Label distribution:")
print(inbreast_metadata['label'].value_counts())
print()
print("View distribution:")
print(inbreast_metadata['view'].value_counts())
print("=" * 80)

In [ ]:
# Debug: Check metadata and filenames
print("=" * 80)
print("METADATA INSPECTION")
print("=" * 80)
print("\nMetadata columns:")
print(list(inbreast_metadata.columns))
print("\nFirst few rows:")
print(inbreast_metadata.head(10))
print()

# Check data types
print("Column data types:")
for col in ['image_id', 'patient_id', 'image_path']:
    if col in inbreast_metadata.columns:
        dtype = inbreast_metadata[col].dtype
        sample_val = inbreast_metadata[col].iloc[0]
        print(f"  {col}: {dtype} (sample: {sample_val}, type: {type(sample_val).__name__})")
print()

# Check if image files actually exist
from pathlib import Path
import glob
img_dir = Path(config.INBREAST_PATH) / config.INBREAST_CONFIG["image_dir"]
print(f"Image directory: {img_dir}")
print(f"Directory exists: {img_dir.exists()}")
print()

# List actual DICOM files in the directory
if img_dir.exists():
    dicom_files = list(img_dir.glob("*.dcm"))
    print(f"Total DICOM files found: {len(dicom_files)}")
    print(f"First 5 DICOM files:")
    for f in dicom_files[:5]:
        print(f"  {f.name}")
    print()

# Try to understand the filename pattern
print("Analyzing filename pattern:")
if img_dir.exists() and dicom_files:
    sample_file = dicom_files[0].name
    print(f"Sample filename: {sample_file}")
    
    # Try to parse: PatientID_Hash_ImageType_Laterality_View_Suffix.dcm
    parts = sample_file.replace('.dcm', '').split('_')
    print(f"Filename parts (split by '_'): {parts}")
    if len(parts) >= 5:
        print(f"  Part 0 (PatientID?): {parts[0]}")
        print(f"  Part 1 (Hash?): {parts[1]}")
        print(f"  Part 2 (ImageType?): {parts[2]}")
        print(f"  Part 3 (Laterality?): {parts[3]}")
        print(f"  Part 4 (View?): {parts[4]}")

print("=" * 80)

In [ ]:
# Fix image_path to match actual image filenames
print("=" * 80)
print("FIXING IMAGE PATHS")
print("=" * 80)
print()

# First, let's see what columns we actually have
print("Available columns in metadata:")
print(list(inbreast_metadata.columns))
print()

from pathlib import Path
import glob
from collections import defaultdict

img_dir = Path(config.INBREAST_PATH) / config.INBREAST_CONFIG["image_dir"]

# Get all image files (check for both DICOM and PNG)
dicom_files = list(img_dir.glob("*.dcm"))
png_files = list(img_dir.glob("*.png"))

print(f"Found {len(dicom_files)} DICOM files")
print(f"Found {len(png_files)} PNG files")

# Determine which format we're using
if len(png_files) > len(dicom_files):
    image_files = png_files
    file_extension = ".png"
    print(f"\nUsing PNG files (converted from DICOM)")
else:
    image_files = dicom_files
    file_extension = ".dcm"
    print(f"\nUsing DICOM files (original)")

print(f"First 5 files:")
for f in image_files[:5]:
    print(f"  {f.name}")
print()

# Parse filenames to extract laterality and view
# Pattern: {PatientID}_{Hash}_{ImageType}_{Laterality}_{View}_{Suffix}.{ext}
# Example: 20586908_6c613a14b80a8591_MG_R_CC_ANON.png

files_by_lat_view = defaultdict(list)
for img_file in image_files:
    parts = img_file.stem.split('_')
    if len(parts) >= 5:
        laterality = parts[3]  # R or L
        view = parts[4]         # CC or ML (not MLO)
        
        # Normalize view (ML -> MLO)
        if view == 'ML':
            view = 'MLO'
        
        key = f"{laterality}_{view}"
        files_by_lat_view[key].append(img_file.name)

print(f"Organized files by laterality+view:")
for key, files in sorted(files_by_lat_view.items()):
    print(f"  {key}: {len(files)} files")
print()

# Now match metadata to files
# Extract laterality from breast_id (e.g., "removed_R" -> "R")
fixed_paths = []
match_count = 0
no_match_count = 0

print("Matching strategy: laterality + view")
print()

for idx, row in inbreast_metadata.iterrows():
    matched_file = None
    
    # Extract laterality from breast_id
    breast_id = str(row.get('breast_id', ''))
    if '_' in breast_id:
        laterality = breast_id.split('_')[-1]  # Get last part (R or L)
    else:
        laterality = None
    
    # Get view
    view = str(row.get('view', '')).upper()
    
    # Build lookup key
    if laterality and view:
        key = f"{laterality}_{view}"
        
        # Get matching files
        matching_files = files_by_lat_view.get(key, [])
        
        if matching_files:
            # Use the index within this laterality+view group
            # Count how many of this type we've already matched
            already_matched = sum(1 for p in fixed_paths if p in matching_files)
            
            # Pick the next available file
            if already_matched < len(matching_files):
                matched_file = matching_files[already_matched]
            else:
                # Reuse files if we run out (shouldn't happen with proper data)
                matched_file = matching_files[0]
    
    if matched_file:
        fixed_paths.append(matched_file)
        match_count += 1
    else:
        fixed_paths.append(None)
        no_match_count += 1
        if no_match_count <= 5:  # Only show first 5 failures
            print(f"⚠ No match for row {idx}: laterality={laterality}, view={view}, key={key if laterality and view else 'N/A'}")

# Update metadata
inbreast_metadata['image_path'] = fixed_paths

# Remove unmatched
original_size = len(inbreast_metadata)
inbreast_metadata = inbreast_metadata[inbreast_metadata['image_path'].notna()].reset_index(drop=True)

print()
print("=" * 80)
print("RESULTS")
print("=" * 80)
print(f"Original dataset size: {original_size} images")
print(f"Successfully matched: {match_count} images")
print(f"Failed to match: {no_match_count} images")
print(f"Final dataset size: {len(inbreast_metadata)} images")
print(f"File format: {file_extension}")
print()
if len(inbreast_metadata) > 0:
    print("Sample matched results:")
    print(inbreast_metadata[['image_id', 'breast_id', 'view', 'label', 'image_path']].head(10))
else:
    print("⚠ No images matched! Check the matching logic above.")
print("=" * 80)

## 4a. Visualize INbreast Dataset (Optional)

Visualize sample INbreast images to verify data loaded correctly.

**Note:** INbreast images should be in PNG format (converted from DICOM).

This shows:
- Sample malignant and benign cases
- Image metadata (view, laterality, BI-RADS)
- Dataset statistics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path
import pandas as pd
import time
import shutil
import os

print("=" * 80)
print("INBREAST DATASET VISUALIZATION (ROBUST VERSION)")
print("=" * 80)
print()

# ============================================================================
# STEP 0: Install pydicom if needed
# ============================================================================
try:
    import pydicom
except:
    print("Installing pydicom...")
    import subprocess
    subprocess.run(["pip", "install", "-q", "pydicom"], check=True)
    import pydicom
    print("✓ pydicom installed")
    print()

# ============================================================================
# STEP 1: Fix metadata paths (.dcm -> .png if needed)
# ============================================================================
print("STEP 1: Fixing image paths...")

# Get image directory
image_dir = Path(config.INBREAST_PATH) / config.INBREAST_CONFIG["image_dir"]

# Check what files exist
try:
    png_files = list(image_dir.glob("*.png"))
    dcm_files = list(image_dir.glob("*.dcm"))
    print(f"  Found {len(png_files)} PNG files, {len(dcm_files)} DCM files")

    # Fix paths if needed
    if len(png_files) > 0 and 'image_path' in inbreast_metadata.columns:
        # Update .dcm to .png in metadata
        inbreast_metadata['image_path'] = inbreast_metadata['image_path'].astype(str).str.replace('.dcm', '.png', case=False, regex=False)
        inbreast_metadata['image_path'] = inbreast_metadata['image_path'].astype(str).str.replace('.DCM', '.png', case=False, regex=False)
        print("  ✓ Updated paths to .png")

except Exception as e:
    print(f"  ⚠ Warning: {str(e)[:50]}")

print()

# ============================================================================
# STEP 2: Create local cache directory (avoids Drive errors)
# ============================================================================
print("STEP 2: Setting up local cache...")

cache_dir = Path("/tmp/inbreast_cache")
cache_dir.mkdir(exist_ok=True, parents=True)
print(f"  Cache directory: {cache_dir}")
print()

# ============================================================================
# STEP 3: Select and cache samples
# ============================================================================
print("STEP 3: Selecting and caching images...")

# Select samples
mal_samples = inbreast_metadata[inbreast_metadata['label'] == 1].head(4)
ben_samples = inbreast_metadata[inbreast_metadata['label'] == 0].head(4)
all_samples = pd.concat([mal_samples, ben_samples]).reset_index(drop=True)

print(f"  Selected: {len(mal_samples)} malignant, {len(ben_samples)} benign")

# Copy to cache with retry
cached_paths = []
success_count = 0

for idx, row in all_samples.iterrows():
    if 'image_path' not in row or pd.isna(row['image_path']):
        cached_paths.append(None)
        continue

    img_name = str(row['image_path'])
    source = image_dir / img_name

    # Try alternative extensions if source doesn't exist
    if not source.exists():
        base = source.stem
        for ext in ['.png', '.PNG', '.dcm', '.DCM']:
            alt = source.parent / (base + ext)
            if alt.exists():
                source = alt
                break

    # Copy to cache with retries
    copied = False
    for attempt in range(3):
        try:
            if source.exists():
                dest = cache_dir / source.name
                if not dest.exists():
                    shutil.copy2(str(source), str(dest))
                cached_paths.append(dest)
                success_count += 1
                copied = True
                break
        except (OSError, IOError) as e:
            if attempt < 2:
                time.sleep(1)
            else:
                print(f"  ⚠ Failed: {source.name}")

    if not copied:
        cached_paths.append(None)

print(f"  ✓ Cached {success_count}/{len(all_samples)} images")
print()

# ============================================================================
# STEP 4: Load and visualize
# ============================================================================
print("STEP 4: Creating visualization...")

def load_img(path):
    """Load image (handles both PNG and DICOM)"""
    try:
        ext = path.suffix.lower()
        if ext in ['.dcm', '.dicom']:
            dcm = pydicom.dcmread(str(path))
            arr = dcm.pixel_array.astype(float)
            if hasattr(dcm, 'PhotometricInterpretation') and dcm.PhotometricInterpretation == "MONOCHROME1":
                arr = arr.max() - arr
            arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8) * 255
            return Image.fromarray(arr.astype(np.uint8), mode='L')
        else:
            return Image.open(path).convert('L')
    except:
        return None

# Create plot
n = len(all_samples)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5*nrows))
if nrows == 1:
    axes = axes.reshape(1, -1)
axes = axes.flatten()

fig.suptitle('INbreast Sample Images', fontsize=16, fontweight='bold')

loaded = 0
failed = 0

for i, (idx, row) in enumerate(all_samples.iterrows()):
    ax = axes[i]

    if i < len(cached_paths) and cached_paths[i] is not None:
        img = load_img(cached_paths[i])

        if img is not None:
            ax.imshow(img, cmap='gray')

            label_txt = 'Malignant' if row['label'] == 1 else 'Benign'
            color = 'red' if row['label'] == 1 else 'green'

            title = f"{label_txt}"
            if 'view' in row and pd.notna(row['view']):
                title += f"\n{row['view']}"

            ax.set_title(title, fontsize=12, fontweight='bold', color=color)
            ax.axis('off')
            loaded += 1
        else:
            ax.text(0.5, 0.5, 'Load failed', ha='center', va='center', fontsize=10)
            ax.axis('off')
            failed += 1
    else:
        ax.text(0.5, 0.5, 'Not cached', ha='center', va='center', fontsize=10)
        ax.axis('off')
        failed += 1

# Hide extra subplots
for i in range(n, len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print(f"✓ Loaded: {loaded}/{n} images")
if failed > 0:
    print(f"⚠ Failed: {failed} images")
print()

# ============================================================================
# STEP 5: Dataset summary
# ============================================================================
print("=" * 80)
print("DATASET SUMMARY")
print("=" * 80)
print(f"Total images: {len(inbreast_metadata)}")

mal = (inbreast_metadata['label'] == 1).sum()
ben = (inbreast_metadata['label'] == 0).sum()
print(f"Malignant: {mal} ({mal/len(inbreast_metadata)*100:.1f}%)")
print(f"Benign: {ben} ({ben/len(inbreast_metadata)*100:.1f}%)")

if 'view' in inbreast_metadata.columns:
    print("\nViews:")
    for v, c in inbreast_metadata['view'].value_counts().items():
        print(f"  {v}: {c}")

print("=" * 80)


## 4b. Convert DICOM to PNG (Required)

**IMPORTANT:** The evaluation pipeline requires PNG images, but INbreast provides DICOM files.

This cell converts all DICOM files to PNG format with proper preprocessing:
- Reads DICOM pixel data
- Applies window/level adjustments if available
- Normalizes to 0-255 range
- Saves as grayscale PNG

**This only needs to be run once.** PNG files will be saved in a new `images_png` directory.

In [ ]:
import pydicom
from PIL import Image
import numpy as np
from pathlib import Path
from tqdm import tqdm
import time

print("=" * 80)
print("DICOM TO PNG CONVERSION (ROBUST VERSION)")
print("=" * 80)
print()

# Set paths - UPDATE THESE IF YOUR STRUCTURE IS DIFFERENT
dicom_dir = Path(INBREAST_PATH) / "AllDICOMs"  # Or use "images" if that's where your DICOMs are
png_output_dir = Path(INBREAST_PATH) / "images_png"

# Create output directory
png_output_dir.mkdir(exist_ok=True)

print(f"Input (DICOM):  {dicom_dir}")
print(f"Output (PNG):   {png_output_dir}")
print()

# Find all DICOM files
dicom_files = list(dicom_dir.glob("*.dcm")) + list(dicom_dir.glob("*.DCM"))
print(f"Found {len(dicom_files)} DICOM files")
print()

if len(dicom_files) == 0:
    print("⚠ No DICOM files found!")
    print(f"  Checked: {dicom_dir}")
    print("  Please update the dicom_dir path above")
else:
    print("Converting DICOM to PNG with retry logic...")
    print()

    conversion_errors = []
    success_count = 0

    for dicom_path in tqdm(dicom_files, desc="Converting"):
        converted = False

        # Try up to 3 times per file (handles Drive disconnections)
        for attempt in range(3):
            try:
                # Read DICOM
                dcm = pydicom.dcmread(str(dicom_path))
                img_array = dcm.pixel_array.astype(float)

                # Apply PhotometricInterpretation if needed
                if hasattr(dcm, 'PhotometricInterpretation'):
                    if dcm.PhotometricInterpretation == "MONOCHROME1":
                        # Invert for MONOCHROME1 (lower values = brighter)
                        img_array = img_array.max() - img_array

                # Normalize to 0-255
                img_min, img_max = img_array.min(), img_array.max()
                if img_max > img_min:
                    img_array = (img_array - img_min) / (img_max - img_min) * 255.0
                else:
                    img_array = np.zeros_like(img_array)

                img_array = img_array.astype(np.uint8)

                # Convert to PIL Image
                img = Image.fromarray(img_array, mode='L')

                # Save as PNG with same filename
                png_filename = dicom_path.stem + ".png"
                png_path = png_output_dir / png_filename

                # Save with retry (handles temporary Drive issues)
                img.save(str(png_path), "PNG")

                success_count += 1
                converted = True
                break  # Success - exit retry loop

            except (OSError, IOError) as e:
                # Handle Drive connection errors
                if "107" in str(e) or "Transport endpoint" in str(e):
                    if attempt < 2:  # Retry
                        time.sleep(2 ** attempt)  # Exponential backoff
                        continue
                    else:
                        conversion_errors.append((dicom_path.name, "Drive connection timeout"))
                else:
                    # Other error - don't retry
                    conversion_errors.append((dicom_path.name, str(e)[:100]))
                    break

            except Exception as e:
                # Unexpected error
                conversion_errors.append((dicom_path.name, str(e)[:100]))
                break

        if not converted:
            # Failed after all retries
            pass

    print()
    print("=" * 80)
    print("CONVERSION COMPLETE")
    print("=" * 80)
    print(f"Total files: {len(dicom_files)}")
    print(f"Successfully converted: {success_count}")
    print(f"Failed: {len(conversion_errors)}")
    print(f"PNG files saved to: {png_output_dir}")
    print()

    if conversion_errors:
        print(f"⚠ Errors: {len(conversion_errors)} files failed")
        print("First 10 errors:")
        for filename, error in conversion_errors[:10]:
            print(f"  {filename}: {error}")
        print()

        if len(conversion_errors) < len(dicom_files) * 0.1:  # Less than 10% failed
            print("✓ Most files converted successfully (>90%)")
            print("  You can proceed with the available images")
        else:
            print("⚠ WARNING: Many files failed to convert")
            print("  This might indicate a Google Drive connection issue")
            print("  Consider remounting Drive and trying again")
    else:
        print("✓ All files converted successfully!")

    print()
    print("⚠ IMPORTANT: Update image directory in config!")
    print(f"  Change 'image_dir' from '{config.INBREAST_CONFIG['image_dir']}' to 'images_png'")
    print("  Run the configuration cell below to update it")
    print("=" * 80)


In [ ]:
# Update config to use PNG images AND fix metadata paths
from pathlib import Path
import pandas as pd

print("=" * 80)
print("POST-CONVERSION SETUP")
print("=" * 80)
print()

# Step 1: Update config.py
print("Step 1: Updating config.py...")
with open('config.py', 'r') as f:
    config_content = f.read()

# Update image directory to point to converted PNGs
updated = False
if '"image_dir": "images"' in config_content:
    config_content = config_content.replace(
        '"image_dir": "images"',
        '"image_dir": "images_png"'
    )
    updated = True
elif '"image_dir": "AllDICOMs"' in config_content:
    config_content = config_content.replace(
        '"image_dir": "AllDICOMs"',
        '"image_dir": "images_png"'
    )
    updated = True

if updated:
    with open('config.py', 'w') as f:
        f.write(config_content)
    print("  ✓ Updated config.py to use images_png directory")
else:
    print("  ✓ Config already set to images_png")

# Step 2: Reload config
import importlib
importlib.reload(config)
print(f"  Current image_dir: {config.INBREAST_CONFIG['image_dir']}")
print()

# Step 3: Fix metadata paths (.dcm -> .png)
print("Step 2: Fixing metadata paths...")
if 'image_path' in inbreast_metadata.columns:
    # Count current extensions
    dcm_count = inbreast_metadata['image_path'].astype(str).str.lower().str.endswith('.dcm').sum()
    png_count = inbreast_metadata['image_path'].astype(str).str.lower().str.endswith('.png').sum()

    print(f"  Current paths: {dcm_count} .dcm, {png_count} .png")

    if dcm_count > 0:
        # Update all .dcm to .png
        inbreast_metadata['image_path'] = inbreast_metadata['image_path'].astype(str).str.replace(
            '.dcm', '.png', case=False, regex=False
        ).str.replace(
            '.DCM', '.PNG', case=False, regex=False
        )

        # Verify
        dcm_after = inbreast_metadata['image_path'].astype(str).str.lower().str.endswith('.dcm').sum()
        png_after = inbreast_metadata['image_path'].astype(str).str.lower().str.endswith('.png').sum()

        print(f"  ✓ Updated to: {dcm_after} .dcm, {png_after} .png")
    else:
        print("  ✓ Paths already use .png extension")
else:
    print("  ⚠ No image_path column in metadata")
print()

# Step 4: Verify files exist
print("Step 3: Verifying files...")
image_dir = Path(config.INBREAST_PATH) / config.INBREAST_CONFIG["image_dir"]

if image_dir.exists():
    png_files = list(image_dir.glob("*.png"))
    print(f"  Found {len(png_files)} PNG files in {image_dir.name}/")

    # Check a few files from metadata
    if len(inbreast_metadata) > 0 and 'image_path' in inbreast_metadata.columns:
        sample_size = min(5, len(inbreast_metadata))
        found_count = 0
        missing_count = 0

        for img_path in inbreast_metadata['image_path'].head(sample_size):
            full_path = image_dir / str(img_path)
            if full_path.exists():
                found_count += 1
            else:
                missing_count += 1
                if missing_count <= 2:  # Show first 2 missing
                    print(f"    ⚠ Not found: {img_path}")

        if missing_count == 0:
            print(f"  ✓ All checked files exist ({found_count}/{sample_size})")
        else:
            print(f"  ⚠ {found_count}/{sample_size} files found, {missing_count} missing")
else:
    print(f"  ⚠ Directory not found: {image_dir}")

print()
print("=" * 80)
print("SETUP COMPLETE")
print("=" * 80)
print(f"Config: {config.INBREAST_CONFIG['image_dir']}")
print(f"Metadata: {len(inbreast_metadata)} images")
if 'image_path' in inbreast_metadata.columns:
    sample_path = str(inbreast_metadata['image_path'].iloc[0])
    print(f"Sample path: {sample_path}")
print()
print("✓ Ready for visualization and evaluation!")
print("=" * 80)


## 5. Select Solution to Evaluate

Choose which Pareto-optimal solution to evaluate on INbreast.

In [ ]:
# ============================================================================
# SELECT WHICH SOLUTION TO EVALUATE
# ============================================================================
# Change this to evaluate different solutions:

solution_id = pareto_df['pr_auc'].idxmax()  # Best PR-AUC (default)
# solution_id = pareto_df['auroc'].idxmax()    # Best AUROC
# solution_id = pareto_df['brier'].idxmin()    # Best Brier
# solution_id = 5                               # Specific solution ID

# ============================================================================

selected_solution = pareto_df.iloc[solution_id]

print("=" * 80)
print(f"SELECTED SOLUTION {solution_id}")
print("=" * 80)
print()
print("Hyperparameters:")
print(f"  Learning rate:          {selected_solution['learning_rate']:.6f}")
print(f"  Weight decay:           {selected_solution['weight_decay']:.6f}")
print(f"  Dropout rate:           {selected_solution['dropout_rate']:.4f}")
print(f"  Augmentation strength:  {selected_solution['augmentation_strength']:.4f}")
print(f"  Unfreeze fraction:      {selected_solution['unfreeze_fraction']:.4f}")
print()
print("VinDr (source) performance:")
print(f"  PR-AUC:     {selected_solution['pr_auc']:.4f}")
print(f"  AUROC:      {selected_solution['auroc']:.4f}")
print(f"  Brier:      {selected_solution['brier']:.4f}")
print(f"  Robustness: {selected_solution['robustness_degradation']:.4f}")
print("=" * 80)

## 6. Load Trained Model

Load the model checkpoint from optimization.

In [ ]:
from pathlib import Path
from models.resnet import ResNet50WithPartialFineTuning

print("=" * 80)
print("LOADING MODEL CHECKPOINT")
print("=" * 80)

# Find checkpoint
checkpoint_path = Path(CHECKPOINT_DIR) / f"eval_{solution_id}" / "best_checkpoint.pt"

if not checkpoint_path.exists():
    print(f"[ERROR] Checkpoint not found: {checkpoint_path}")
    print()
    print("Available checkpoint directories:")
    for d in sorted(Path(CHECKPOINT_DIR).glob("eval_*")):
        print(f"  {d.name}")
    raise FileNotFoundError(f"Checkpoint not found for solution {solution_id}")

print(f"Loading: {checkpoint_path}")
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
print(f"[OK] Checkpoint loaded")
print(f"  Epoch: {checkpoint.get('epoch', 'unknown')}")
print(f"  Best PR-AUC: {checkpoint.get('best_pr_auc', 'unknown')}")
print()

# Create model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = ResNet50WithPartialFineTuning(
    dropout_rate=selected_solution['dropout_rate'],
    unfreeze_fraction=selected_solution['unfreeze_fraction'],
)

model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print("[OK] Model created and weights loaded")
print("=" * 80)

## 7. Run Zero-Shot Inference

Evaluate the model on INbreast **without any fine-tuning**.

In [ ]:
from data.dataset import MammographyDataset, get_base_transform
import torch

# Create INbreast dataloader (no augmentation for evaluation)
inbreast_image_dir = str(Path(config.INBREAST_PATH) / config.INBREAST_CONFIG["image_dir"])

# Use NUM_WORKERS if defined, otherwise default to 2 for Colab
num_workers = getattr(config, 'NUM_WORKERS', 2)

# Create validation dataset directly (no training, so skip train loader)
base_transform = get_base_transform()
val_dataset = MammographyDataset(
    metadata=inbreast_metadata,
    image_dir=inbreast_image_dir,
    transform=base_transform,
    augmentation=None,  # No augmentation for evaluation
)

# Create dataloader
inbreast_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

print(f"[OK] Created dataloader")
print(f"  Total batches: {len(inbreast_loader)}")
print(f"  Batch size: {config.BATCH_SIZE}")
print(f"  Num workers: {num_workers}")
print(f"  Total images: {len(inbreast_metadata)}")

In [ ]:
import numpy as np

print("=" * 80)
print("RUNNING ZERO-SHOT INFERENCE ON INBREAST")
print("=" * 80)
print()

all_preds = []
all_labels = []
all_image_ids = []

with torch.no_grad():
    for batch_idx, (images, labels, image_ids) in enumerate(inbreast_loader):
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass (model already returns probabilities with sigmoid)
        probs = model(images)

        # Store predictions
        all_preds.extend(probs.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())
        all_image_ids.extend(image_ids)  # image_ids is already a list

        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(inbreast_loader):
            print(f"  Processed {batch_idx + 1}/{len(inbreast_loader)} batches", end='\r')

print(f"\n[OK] Inference complete: {len(all_preds)} images")
print("=" * 80)

## 8. Evaluate Performance

Compute metrics at both image-level and breast-level.

In [ ]:
from training.metrics import compute_metrics

print("=" * 80)
print("IMAGE-LEVEL EVALUATION")
print("=" * 80)

all_preds_array = np.array(all_preds)
all_labels_array = np.array(all_labels)

image_metrics = compute_metrics(all_preds_array, all_labels_array)

print(f"\nImages evaluated: {len(all_preds)}")
print()
print("Metrics:")
print(f"  PR-AUC:        {image_metrics['pr_auc']:.4f}")
print(f"  AUROC:         {image_metrics['auroc']:.4f}")
print(f"  Brier Score:   {image_metrics['brier']:.4f}")
print(f"  Brier Null:    {image_metrics['brier_null']:.4f}")
print(f"  Scaled Brier:  {image_metrics['scaled_brier']:.4f}")
print("=" * 80)

In [ ]:
from utils.noisy_or import aggregate_to_breast_level

print("=" * 80)
print("BREAST-LEVEL EVALUATION (NOISY OR AGGREGATION)")
print("=" * 80)

# Create image predictions dictionary
image_predictions = {img_id: pred for img_id, pred in zip(all_image_ids, all_preds)}

# Aggregate to breast level
breast_preds, breast_labels = aggregate_to_breast_level(
    image_predictions=image_predictions,
    metadata=inbreast_metadata
)

print(f"\nAggregated to {len(breast_preds)} breasts")
print()

# Compute breast-level metrics
breast_metrics = compute_metrics(breast_preds, breast_labels)

print("Metrics:")
print(f"  PR-AUC:        {breast_metrics['pr_auc']:.4f}")
print(f"  AUROC:         {breast_metrics['auroc']:.4f}")
print(f"  Brier Score:   {breast_metrics['brier']:.4f}")
print(f"  Brier Null:    {breast_metrics['brier_null']:.4f}")
print(f"  Scaled Brier:  {breast_metrics['scaled_brier']:.4f}")
print("=" * 80)

## 9. Compare Source vs Target Performance

Analyze transfer learning effectiveness.

In [ ]:
print("=" * 80)
print("TRANSFER LEARNING ANALYSIS")
print("=" * 80)
print()
print(f"Model: Solution {solution_id}")
print()
print("Source Dataset (VinDr-Mammo):")
print(f"  PR-AUC: {selected_solution['pr_auc']:.4f}")
print(f"  AUROC:  {selected_solution['auroc']:.4f}")
print(f"  Brier:  {selected_solution['brier']:.4f}")
print()
print("Target Dataset (INbreast) - Image Level:")
print(f"  PR-AUC: {image_metrics['pr_auc']:.4f} ({(image_metrics['pr_auc'] / selected_solution['pr_auc'] - 1) * 100:+.1f}%)")
print(f"  AUROC:  {image_metrics['auroc']:.4f} ({(image_metrics['auroc'] / selected_solution['auroc'] - 1) * 100:+.1f}%)")
print(f"  Brier:  {image_metrics['brier']:.4f} ({(image_metrics['brier'] / selected_solution['brier'] - 1) * 100:+.1f}%)")
print()
print("Target Dataset (INbreast) - Breast Level:")
print(f"  PR-AUC: {breast_metrics['pr_auc']:.4f} ({(breast_metrics['pr_auc'] / selected_solution['pr_auc'] - 1) * 100:+.1f}%)")
print(f"  AUROC:  {breast_metrics['auroc']:.4f} ({(breast_metrics['auroc'] / selected_solution['auroc'] - 1) * 100:+.1f}%)")
print(f"  Brier:  {breast_metrics['brier']:.4f} ({(breast_metrics['brier'] / selected_solution['brier'] - 1) * 100:+.1f}%)")
print()

# Transfer quality assessment
transfer_ratio = breast_metrics['pr_auc'] / selected_solution['pr_auc']
print("Transfer Quality Assessment:")
if transfer_ratio >= 0.9:
    print("  ✓ EXCELLENT - Minimal performance degradation (<10%)")
elif transfer_ratio >= 0.8:
    print("  ✓ GOOD - Acceptable performance retention (>80%)")
elif transfer_ratio >= 0.7:
    print("  ⚠ MODERATE - Noticeable performance drop (70-80%)")
else:
    print("  ⚠ POOR - Significant performance degradation (<70%)")

print("=" * 80)

## 10. Save Results

Save evaluation results to Google Drive.

In [ ]:
from datetime import datetime
import json

print("=" * 80)
print("SAVING EVALUATION RESULTS")
print("=" * 80)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create evaluation results directory
eval_dir = Path(OPTIMIZATION_DIR) / "inbreast_evaluation"
eval_dir.mkdir(exist_ok=True)

# Prepare results dictionary
eval_results = {
    "timestamp": timestamp,
    "solution_id": int(solution_id),
    "hyperparameters": {
        "learning_rate": float(selected_solution['learning_rate']),
        "weight_decay": float(selected_solution['weight_decay']),
        "dropout_rate": float(selected_solution['dropout_rate']),
        "augmentation_strength": float(selected_solution['augmentation_strength']),
        "unfreeze_fraction": float(selected_solution['unfreeze_fraction']),
    },
    "source_performance": {
        "dataset": "VinDr-Mammo",
        "pr_auc": float(selected_solution['pr_auc']),
        "auroc": float(selected_solution['auroc']),
        "brier": float(selected_solution['brier']),
        "robustness_degradation": float(selected_solution['robustness_degradation']),
    },
    "target_performance": {
        "dataset": "INbreast",
        "image_level": {
            "n_images": len(all_preds),
            "pr_auc": float(image_metrics['pr_auc']),
            "auroc": float(image_metrics['auroc']),
            "brier": float(image_metrics['brier']),
            "brier_null": float(image_metrics['brier_null']),
            "scaled_brier": float(image_metrics['scaled_brier']),
        },
        "breast_level": {
            "n_breasts": len(breast_preds),
            "pr_auc": float(breast_metrics['pr_auc']),
            "auroc": float(breast_metrics['auroc']),
            "brier": float(breast_metrics['brier']),
            "brier_null": float(breast_metrics['brier_null']),
            "scaled_brier": float(breast_metrics['scaled_brier']),
        },
    },
}

# Save JSON
json_path = eval_dir / f"evaluation_solution_{solution_id}_{timestamp}.json"
with open(json_path, 'w') as f:
    json.dump(eval_results, f, indent=2)
print(f"[OK] Saved results: {json_path}")

# Save predictions CSV
predictions_df = pd.DataFrame({
    'image_id': all_image_ids,
    'prediction': all_preds,
    'label': all_labels,
})
csv_path = eval_dir / f"predictions_solution_{solution_id}_{timestamp}.csv"
predictions_df.to_csv(csv_path, index=False)
print(f"[OK] Saved predictions: {csv_path}")

print()
print(f"Results saved to: {eval_dir}")
print("=" * 80)

## Summary

Zero-shot evaluation complete! Key findings are displayed above.

**Next Steps:**
- Evaluate other Pareto solutions (change `solution_id` in Section 5)
- Compare transfer performance across all solutions
- Analyze which hyperparameters lead to better transfer
- Consider ensemble methods combining multiple solutions

**All results saved to Google Drive:**
- JSON file with all metrics
- CSV file with image-level predictions

Access anytime at: `/content/drive/MyDrive/vindr_optimization/inbreast_evaluation/`